In [1]:
import scanpy as sc
import liana

/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(ms

## Load data

In [62]:
slide = "Visium_18"
pseudocell_spatial = sc.read_h5ad(f"../results/cardiomyocyte_subtypes/cell2location_map/spot_celltype_pseudobulks_{slide}.h5ad")
pseudocell_spatial.var["feature_name"] = pseudocell_spatial.var_names
pseudocell_spatial.var_names = pseudocell_spatial.var["SYMBOL"]

In [63]:
pseudocell_spatial

AnnData object with n_obs × n_vars = 25004 × 11613
    obs: 'spot', 'cell_type', 'molecular_niche', 'celltype_niche'
    var: 'feature_types', 'genome', 'SYMBOL', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'rps', 'mrp', 'rpl', 'duplicated', 'feature_name'
    obsm: 'spatial'

In [64]:
pseudocell_spatial.var.head(2)

,feature_types,genome,SYMBOL,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,mt,rps,mrp,rpl,duplicated,feature_name
SYMBOL,,,,,,,,,,,,,,,
DPM1,Gene Expression,GRCh38,DPM1,931,0.314804,0.273688,73.994413,1127.0,7.028202,False,False,False,False,False,ENSG00000000419
SCYL3,Gene Expression,GRCh38,SCYL3,256,0.075698,0.072970,92.849162,271.0,5.605802,False,False,False,False,False,ENSG00000000457


# Infer cell communication for both niche 1 and niche 2

### Niche 1

In [65]:
# Cut by molecular niches
rna_niche_1 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_1", :]

liana.method.cellphonedb(
    rna_niche_1, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


4 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.53 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 12348 samples and 11609 features


100%|██████████| 1000/1000 [00:23<00:00, 42.45it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


### Format network for ReCoN

In [66]:
ccc_layer_niche_1 = rna_niche_1.uns["cpdb_res"].copy()
ccc_layer_niche_1 = ccc_layer_niche_1[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_1.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_1["celltype_source"] = rna_niche_1.uns["cpdb_res"]["source"]
ccc_layer_niche_1["celltype_target"] = rna_niche_1.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_1 = ccc_layer_niche_1.loc[~ccc_layer_niche_1.isna().sum(1).astype(bool), :]#.shape

### Save network

In [67]:
ccc_layer_niche_1.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_1_{slide}.csv")

### Niche 5

### Format network for ReCoN

In [68]:
# Cut by molecular niches
rna_niche_5 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_5", :]

liana.method.cellphonedb(
    rna_niche_5, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


201 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.54 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 3297 samples and 11412 features


100%|██████████| 1000/1000 [00:06<00:00, 163.78it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


In [69]:
ccc_layer_niche_5 = rna_niche_5.uns["cpdb_res"].copy()
ccc_layer_niche_5 = ccc_layer_niche_5[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_5.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_5["celltype_source"] = rna_niche_5.uns["cpdb_res"]["source"]
ccc_layer_niche_5["celltype_target"] = rna_niche_5.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_5 = ccc_layer_niche_5.loc[~ccc_layer_niche_5.isna().sum(1).astype(bool), :]#.shape

### Save network

In [70]:
ccc_layer_niche_5.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_5_{slide}.csv")

### Niche 2

In [71]:
# Cut by molecular niches
rna_niche_2 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_2", :]

liana.method.cellphonedb(
    rna_niche_2, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format
2356 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.63 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 168 samples and 9257 features


100%|██████████| 1000/1000 [00:01<00:00, 558.66it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


In [72]:
ccc_layer_niche_2 = rna_niche_2.uns["cpdb_res"].copy()
ccc_layer_niche_2 = ccc_layer_niche_2[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_2.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_2["celltype_source"] = rna_niche_2.uns["cpdb_res"]["source"]
ccc_layer_niche_2["celltype_target"] = rna_niche_2.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_2 = ccc_layer_niche_2.loc[~ccc_layer_niche_2.isna().sum(1).astype(bool), :]#.shape

### Save network

In [73]:
ccc_layer_niche_5.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_2_{slide}.csv")

### Niche 4

In [74]:
# Cut by molecular niches
rna_niche_4 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_4", :]
liana.method.cellphonedb(
    rna_niche_4, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


72 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.53 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 4788 samples and 11541 features


100%|██████████| 1000/1000 [00:07<00:00, 132.85it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


### Format network for ReCoN

In [75]:
ccc_layer_niche_4 = rna_niche_4.uns["cpdb_res"].copy()
ccc_layer_niche_4 = ccc_layer_niche_4[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_4.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_4["celltype_source"] = rna_niche_4.uns["cpdb_res"]["source"]
ccc_layer_niche_4["celltype_target"] = rna_niche_4.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_4 = ccc_layer_niche_4.loc[~ccc_layer_niche_4.isna().sum(1).astype(bool), :]#.shape

### Save network

In [76]:
ccc_layer_niche_4.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_4_{slide}.csv")

### Cell type average expression - pseudobulk

In [ ]:
import pandas as pd
import numpy as np

# Expression matrix (n_obs × n_vars) as a dense array
# You could also work with sparse, but let's make it dense for mean calculation
X = rna_niche_1.X

# If it's sparse, convert for mean computation
if not isinstance(X, np.ndarray):
    X = X.toarray()

# Grouping key
group_key = "cell_type"  # from adata.obs

# Convert to DataFrame for grouping
expr_df_niche_1 = pd.DataFrame(X, index=rna_niche_1.obs_names, columns=rna_niche_1.var_names)
expr_df_niche_1[group_key] = rna_niche_1.obs[group_key].values

# Compute average per cell type
avg_expr_niche_1 = expr_df_niche_1.groupby(group_key).mean()
avg_expr_niche_1
# avg_expr is now: rows = cell types, columns = genes

/local/scratch/tmp/ipykernel_1814658/1427621281.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


SYMBOL,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,AC097526.1,AC083862.3,AL513487.1,AC006230.1,AL390242.1,AC011407.1,AL953883.1,Z85996.3,AL353135.2,AL592295.5
cell_type,,,,,,,,,,,,,,,,,,,,,
Cardiomyocyte,0.028304,0.005843,0.002048,0.000048,0.002183,0.062428,0.016094,0.003974,0.003713,0.011727,...,0.004893,0.001551,0.003519,0.000166,0.000921,0.000074,0.011123,2.010850e-03,0.000220,0.000580
Endothelial,0.003618,0.000686,0.000281,0.000671,0.011822,0.005134,0.001935,0.000757,0.000968,0.005222,...,0.000032,0.000317,0.000018,0.000127,0.000010,0.001904,0.000034,1.072618e-06,0.000005,0.000045
Fibroblast,0.005198,0.001547,0.000630,0.000049,0.125510,0.009984,0.001628,0.000836,0.000744,0.001997,...,0.000029,0.000078,0.000020,0.000147,0.000040,0.000196,0.000013,9.521106e-07,0.000017,0.000161
Lymphoid,0.002573,0.000966,0.000456,0.000645,0.007965,0.001754,0.001439,0.000512,0.000129,0.005901,...,0.000237,0.000047,0.000021,0.000014,0.000006,0.000057,0.000017,7.197556e-07,0.000002,0.000018
Myeloid,0.004423,0.000971,0.000775,0.005581,0.002060,0.014922,0.004258,0.001189,0.000315,0.001721,...,0.000063,0.000107,0.000022,0.000616,0.000014,0.000005,0.000008,1.718119e-06,0.000002,0.000109
Pericyte,0.003464,0.000845,0.000467,0.000016,0.002899,0.006930,0.001438,0.000788,0.000141,0.000940,...,0.000129,0.000029,0.000016,0.000002,0.000016,0.000972,0.000045,1.349906e-06,0.000020,0.000031
vSMCs,0.004957,0.001211,0.000522,0.000044,0.019794,0.009022,0.001006,0.000977,0.000220,0.000992,...,0.000339,0.000087,0.000024,0.000084,0.000018,0.001527,0.000061,2.619890e-06,0.000012,0.000093


In [ ]:
import pandas as pd
import numpy as np

# Expression matrix (n_obs × n_vars) as a dense array
# You could also work with sparse, but let's make it dense for mean calculation
X = rna_niche_5.X

# If it's sparse, convert for mean computation
if not isinstance(X, np.ndarray):
    X = X.toarray()

# Grouping key
group_key = "cell_type"  # from adata.obs

# Convert to DataFrame for grouping
expr_df_niche_5 = pd.DataFrame(X, index=rna_niche_5.obs_names, columns=rna_niche_5.var_names)
expr_df_niche_5[group_key] = rna_niche_5.obs[group_key].values

# Compute average per cell type
avg_expr_niche_5 = expr_df_niche_5.groupby(group_key).mean()
avg_expr_niche_5
# avg_expr is now: rows = cell types, columns = genes

/local/scratch/tmp/ipykernel_1814658/3687922343.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


SYMBOL,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,AC097526.1,AC083862.3,AL513487.1,AC006230.1,AL390242.1,AC011407.1,AL953883.1,Z85996.3,AL353135.2,AL592295.5
cell_type,,,,,,,,,,,,,,,,,,,,,
Cardiomyocyte,0.038282,0.006904,0.003556,0.000086,0.002059,0.106144,0.018049,0.006357,0.004774,0.007645,...,0.001291,0.002190,0.0,0.000213,0.0,0.000035,0.011861,0.008510,0.000760,0.000302
Endothelial,0.004923,0.000824,0.000488,0.001206,0.011107,0.008707,0.002179,0.001205,0.001255,0.003401,...,0.000008,0.000445,0.0,0.000164,0.0,0.000903,0.000038,0.000005,0.000014,0.000023
Fibroblast,0.007069,0.001863,0.001089,0.000089,0.117957,0.016914,0.001827,0.001332,0.000963,0.001295,...,0.000007,0.000109,0.0,0.000189,0.0,0.000093,0.000015,0.000004,0.000053,0.000083
Lymphoid,0.003499,0.001158,0.000791,0.001152,0.007476,0.002969,0.001613,0.000814,0.000166,0.003824,...,0.000061,0.000066,0.0,0.000018,0.0,0.000027,0.000019,0.000003,0.000006,0.000009
Myeloid,0.006022,0.001169,0.001339,0.010013,0.001937,0.025303,0.004789,0.001899,0.000407,0.001119,...,0.000016,0.000150,0.0,0.000792,0.0,0.000002,0.000009,0.000007,0.000007,0.000056
Pericyte,0.004698,0.001018,0.000807,0.000030,0.002725,0.011764,0.001617,0.001254,0.000184,0.000611,...,0.000034,0.000041,0.0,0.000003,0.0,0.000459,0.000050,0.000006,0.000064,0.000016
vSMCs,0.006746,0.001446,0.000907,0.000080,0.018611,0.015264,0.001128,0.001561,0.000283,0.000644,...,0.000086,0.000122,0.0,0.000108,0.0,0.000717,0.000066,0.000011,0.000039,0.000048


In [ ]:
import pandas as pd
import numpy as np

# Expression matrix (n_obs × n_vars) as a dense array
# You could also work with sparse, but let's make it dense for mean calculation
X = rna_niche_4.X

# If it's sparse, convert for mean computation
if not isinstance(X, np.ndarray):
    X = X.toarray()

# Grouping key
group_key = "cell_type"  # from adata.obs

# Convert to DataFrame for grouping
expr_df_niche_4 = pd.DataFrame(X, index=rna_niche_4.obs_names, columns=rna_niche_4.var_names)
expr_df_niche_4[group_key] = rna_niche_4.obs[group_key].values

# Compute average per cell type
avg_expr_niche_4 = expr_df_niche_4.groupby(group_key).mean()
avg_expr_niche_4
# avg_expr is now: rows = cell types, columns = genes

/local/scratch/tmp/ipykernel_1814658/3019780622.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


SYMBOL,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,AC097526.1,AC083862.3,AL513487.1,AC006230.1,AL390242.1,AC011407.1,AL953883.1,Z85996.3,AL353135.2,AL592295.5
cell_type,,,,,,,,,,,,,,,,,,,,,
Cardiomyocyte,0.032920,0.008418,0.004017,0.000091,0.002849,0.097821,0.015759,0.005148,0.004196,0.009483,...,0.0,0.003488,0.008228,0.000252,0.001051,0.000053,0.017820,0.0,0.0,0.000543
Endothelial,0.004205,0.000988,0.000552,0.001263,0.015423,0.008041,0.001907,0.000976,0.001101,0.004200,...,0.0,0.000717,0.000043,0.000195,0.000010,0.001341,0.000055,0.0,0.0,0.000042
Fibroblast,0.006035,0.002226,0.001236,0.000093,0.163814,0.015665,0.001605,0.001081,0.000853,0.001606,...,0.0,0.000176,0.000049,0.000225,0.000042,0.000138,0.000022,0.0,0.0,0.000152
Lymphoid,0.002977,0.001385,0.000894,0.001204,0.010383,0.002745,0.001416,0.000664,0.000146,0.004742,...,0.0,0.000106,0.000049,0.000021,0.000006,0.000041,0.000027,0.0,0.0,0.000017
Myeloid,0.005135,0.001398,0.001525,0.010515,0.002688,0.023395,0.004200,0.001539,0.000359,0.001387,...,0.0,0.000242,0.000054,0.000938,0.000015,0.000003,0.000013,0.0,0.0,0.000102
Pericyte,0.004018,0.001216,0.000918,0.000031,0.003781,0.010886,0.001417,0.001013,0.000161,0.000755,...,0.0,0.000066,0.000039,0.000003,0.000016,0.000690,0.000073,0.0,0.0,0.000028
vSMCs,0.005766,0.001751,0.001028,0.000084,0.025849,0.014142,0.000992,0.001258,0.000252,0.000801,...,0.0,0.000196,0.000056,0.000130,0.000020,0.001076,0.000099,0.0,0.0,0.000089


## Save average expression per cell type in each niche, as input for the exploration

In [ ]:
avg_expr_niche_1.to_csv(f"../results/cardiomyocyte_subtypes/avg_expression_niche_1_{slide}.csv")

In [ ]:
avg_expr_niche_5.to_csv(f"../results/cardiomyocyte_subtypes/avg_expression_niche_5_{slide}.csv")

In [ ]:
avg_expr_niche_4.to_csv(f"../results/cardiomyocyte_subtypes/avg_expression_niche_4_{slide}.csv")